In [ ]:
import sys, os
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

In [ ]:
%LoadCheckpoint /home/colinc/code/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/annotated_cpu/checkpoints/post_cell_30.pickle

In [ ]:
%%RecordEvent
%%time
### cell 31 ###

# Preprocessing: define question and x-axis title
question_of_interest_cell43 = 'For how many years have you used machine learning methods?'
title_for_x_axis_cell43 = ''

# Rename the 2018 column and normalize categories in 2018 and 2019
responses_df_2018_cell10.rename(
    columns={
        'For how many years have you used machine learning methods (at work or in school)?': question_of_interest_cell43
    }, inplace=True
)
# Apply all replacements in one call per DataFrame
responses_df_2018_cell10[question_of_interest_cell43].replace({
    '< 1 year': 'Under 1 year',
    '10-15 years': '10-20 years',
    '20+ years': '10-20 years',
    'I have never studied machine learning but plan to learn in the future': 'I do not use machine learning methods',
    'I have never studied machine learning and I do not plan to': 'I do not use machine learning methods',
}, inplace=True)
responses_df_2019_cell10[question_of_interest_cell43].replace({
    '< 1 years': 'Under 1 year',
    '10-15 years': '10-20 years',
    '20+ years': '20 or more years',
}, inplace=True)

# Optimized functions
def count_then_return_percent_43(df, x_axis_title):
    """
    Return percentage distribution of values in df[x_axis_title], including NaNs, rounded to one decimal place.
    """
    s = df[x_axis_title]
    total = s.count()
    return s.value_counts(dropna=False).mul(100).div(total).round(1)


def combine_subset_of_data_from_multiple_years_43(question_of_interest, x_axis_title, include_2017=None):
    """
    For each year’s responses DataFrame, compute percentage distribution of question_of_interest,
    add 'year' and the x-axis column, then concatenate all years into one DataFrame.
    If include_2017 is not None, include the 2017 DataFrame as well.
    """
    # List of (year, DataFrame) tuples in ascending year order
    year_dfs = [
        (2018, responses_df_2018_cell10),
        (2019, responses_df_2019_cell10),
        (2020, responses_df_2020),
        (2021, responses_df_2021),
        (2022, responses_df_2022_cell10),
    ]
    if include_2017 is not None:
        year_dfs.insert(0, (2017, responses_df_2017))

    parts = []
    for year, df in year_dfs:
        pct = count_then_return_percent_43(df, question_of_interest).sort_index()
        tmp = (
            pct.rename_axis(x_axis_title)
               .reset_index(name='percentage')
               .assign(year=str(year))
        )
        parts.append(tmp[['percentage', 'year', x_axis_title]])

    return pd.concat(parts, ignore_index=True)

# Call the optimized combine function and sort
ml_exp_df_combined = (
    combine_subset_of_data_from_multiple_years_43(
        question_of_interest_cell43,
        title_for_x_axis_cell43
    )
    .sort_values(by=['year', 'percentage'], ascending=True)
)
ml_exp_df_combined.info()

In [ ]:
%Checkpoint /home/colinc/code/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/rewritten_cpu/o4_mini_high/checkpoints/post_cell_31_try_2.pickle

In [ ]:
%PrintCellInfo opt_cell_exec_info

In [ ]:

with open("/home/colinc/code/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/opt_cell_exec_info_31_try_2.pkl", "wb") as f:
    pickle.dump(opt_cell_exec_info[31], f)


In [ ]:
opt_output = Out.get(4)